# Frequency response and Bode plots: notebook

The running example of the Note: the robot's PD heading loop, yaw inertia I = 0.1 kg·m², gains Kp = 0.4 N·m/rad and Kd = 0.08 N·m·s/rad (the derivative acts on the measured turn rate), so the command-to-heading transfer function is G(s) = 4/(s² + 0.8 s + 4). The speed plant 1/(8 s + 2) (mass 8 kg, drag 2 N·s/m) is the one-pole example. We check every number in the Note: by the formula G(jω) and by simulating the loop on a 1 ms grid and fitting a sine to the settled heading.

In [1]:
import numpy as np
from scipy.optimize import brentq

M, B, I = 8.0, 2.0, 0.1            # mass (kg), drag (N s/m), yaw inertia (kg m^2)
KP, KD, TAU_MAX = 0.4, 0.08, 0.2   # PD gains, torque limit (N m)

def G(s, kp=KP, kd=KD):            # heading command -> heading
    return kp / (I * s**2 + kd * s + kp)

def P_speed(s):                    # drive force (N) -> speed (m/s)
    return 1 / (M * s + B)

db = lambda x: 20 * np.log10(np.abs(x))
deg = lambda x: np.degrees(np.angle(x))

def simulate(t, ref, kp=KP, kd=KD):
    """I th'' = torque, torque = kp (ref - th) - kd w, clipped at 0.2 N m (semi-implicit Euler, 1 ms)."""
    dt = t[1] - t[0]; th = np.zeros(len(t)); w = np.zeros(len(t)); tq = np.zeros(len(t))
    for i in range(len(t) - 1):
        tq[i] = np.clip(kp * (ref[i] - th[i]) - kd * w[i], -TAU_MAX, TAU_MAX)
        w[i + 1] = w[i] + dt * tq[i] / I
        th[i + 1] = th[i] + dt * w[i + 1]
    return th, tq

def sine_fit(t, y, w):
    A = np.column_stack([np.sin(w * t), np.cos(w * t), np.ones_like(t)])
    a, b, _ = np.linalg.lstsq(A, y, rcond=None)[0]
    return np.hypot(a, b), np.degrees(np.arctan2(b, a))

wn, zeta = np.sqrt(KP / I), KD / (2 * np.sqrt(KP * I))
print("natural frequency", wn, "rad/s, damping ratio", round(zeta, 3))

natural frequency 2.0 rad/s, damping ratio 0.2


## 1. The weave at three rates (Note, Section 1)
Heading command 0.1 sin(ωt). Formula and simulation from rest (second half of a long run).

In [2]:
A = 0.1
for w in [0.5, 1.92, 5.0]:
    t = np.arange(0, max(80, 40 * 2 * np.pi / w), 1e-3)
    th, tq = simulate(t, A * np.sin(w * t))
    k = t > t[-1] / 2
    amp, ph = sine_fit(t[k], th[k], w)
    g = G(1j * w)
    print(f"w = {w}: formula gain {abs(g):.3f}, phase {deg(g):.1f} deg | simulated gain {amp / A:.3f}, phase {ph:.1f} deg"
          f" | heading swing {A * abs(g):.3f} rad, largest torque {np.abs(tq).max():.3f} N m")

w = 0.5: formula gain 1.061, phase -6.1 deg | simulated gain 1.061, phase -6.1 deg | heading swing 0.106 rad, largest torque 0.007 N m


w = 1.92: formula gain 2.552, phase -78.5 deg | simulated gain 2.551, phase -78.4 deg | heading swing 0.255 rad, largest torque 0.094 N m


w = 5.0: formula gain 0.187, phase -169.2 deg | simulated gain 0.187, phase -169.2 deg | heading swing 0.019 rad, largest torque 0.057 N m


## 2. A sine through the speed loop's PI controller of RO-060, Kp = 20 N per m/s, Ki = 10 N per m (Note, Section 2.1)
The speed error swings 0.1 m/s at 0.5 rad/s.

In [3]:
KPV, KIV, w, E = 20.0, 10.0, 0.5, 0.1
t = np.linspace(0, 50, 50001)
e = E * np.sin(w * t)
running = np.concatenate([[0], np.cumsum((e[1:] + e[:-1]) / 2 * np.diff(t))])   # running total of e, in m
print("running total: mean", round(running.mean(), 3), "m, swing amplitude", round((running.max() - running.min()) / 2, 3), "m")
u = KPV * e + KIV * (running - E / w)          # leave out the fixed level E/w = 0.2 m
amp, ph = sine_fit(t, u, w)
C = KPV + KIV / (1j * w)
print("proportional part", KPV * E, "N; integral swing", KIV * E / w, "N; output amplitude", round(amp, 3), "N, phase", round(ph, 2), "deg")
print("C(0.5j) =", C, " gain", round(abs(C), 3), "N per m/s, phase", round(deg(C), 1), "deg, delay", round(-np.angle(C) / w, 3), "s")

running total: mean 0.201 m, swing amplitude 0.2 m
proportional part 2.0 N; integral swing 2.0 N; output amplitude 2.828 N, phase -45.0 deg
C(0.5j) = (20-20j)  gain 28.284 N per m/s, phase -45.0 deg, delay 1.571 s


## 3. The weave at 1 rad/s, simulated from rest (Note, Section 2.2)

In [4]:
w, A = 1.0, 0.1
t = np.arange(0, 30.0001, 1e-3)
th, tq = simulate(t, A * np.sin(w * t))
amp, ph = sine_fit(t[t >= 15], th[t >= 15], w)
g = G(1j * w)
print("simulated gain", round(amp / A, 4), "phase", round(ph, 2), "deg;  formula", round(abs(g), 4), round(deg(g), 2))
print("time lag", round(-np.radians(deg(g)) / w, 3), "s of a period", round(2 * np.pi / w, 3), "s;  phase from the lag",
      round(-360 * (-np.radians(deg(g)) / w) / (2 * np.pi / w), 2), "deg")
left = np.abs(th - A * abs(g) * np.sin(w * t + np.angle(g)))
print("start-up part: largest", round(left[t < 3].max(), 4), "rad before 3 s, largest after 10 s", round(left[t >= 10].max(), 4))
print("decay of the start-up swing exp(-0.4 t) at t = 10 s:", round(np.exp(-0.4 * 10), 3))

simulated gain 1.2883 phase -14.93 deg;  formula 1.2883 -14.93
time lag 0.261 s of a period 6.283 s;  phase from the lag -14.93 deg
start-up part: largest 0.0437 rad before 3 s, largest after 10 s 0.0009
decay of the start-up swing exp(-0.4 t) at t = 10 s: 0.018


## 4. Gain and phase from G(jω) by hand (Note, Section 3)

In [5]:
print("speed plant at 0.25 rad/s:", P_speed(0.25j), "gain", round(abs(P_speed(0.25j)), 4), "phase", deg(P_speed(0.25j)),
      "lag", round(np.radians(45) / 0.25, 3), "s")
den = KP - I * 1**2 + KD * 1j
print("heading loop at 1 rad/s: denominator", den, " |den|", round(abs(den), 4), " gain", round(KP / abs(den), 4),
      " phase", round(-np.degrees(np.arctan2(0.08, 0.3)), 2))

speed plant at 0.25 rad/s: (0.25-0.25j) gain 0.3536 phase -45.0 lag 3.142 s
heading loop at 1 rad/s: denominator (0.30000000000000004+0.08j)  |den| 0.3105  gain 1.2883  phase -14.93


## 5. Decibels (Note, Section 4.1)

In [6]:
for x in [10, 2.55, 2.5, 1.288, 1, 1 / np.sqrt(2), 0.5, 0.354, 0.1, 0.01]:
    print(f"{x:.4g} -> {db(x):.2f} dB")
print("0.5 x 0.707: ", round(db(0.5) + db(1 / np.sqrt(2)), 2), "dB =", round(db(0.5 / np.sqrt(2)), 2), "dB")

10 -> 20.00 dB
2.55 -> 8.13 dB
2.5 -> 7.96 dB
1.288 -> 2.20 dB
1 -> 0.00 dB
0.7071 -> -3.01 dB
0.5 -> -6.02 dB
0.354 -> -9.02 dB
0.1 -> -20.00 dB
0.01 -> -40.00 dB
0.5 x 0.707:  -9.03 dB = -9.03 dB


## 6. Asymptotes (Note, Section 5)

In [7]:
for w in [0.025, 0.25, 2.5, 25]:
    p = P_speed(1j * w)
    print(f"speed plant at {w}: exact {db(p):.2f} dB, phase {deg(p):.1f} deg")
for w in [0.2, 2, 20, 200]:
    g = G(1j * w)
    print(f"heading loop at {w}: exact {db(g):.2f} dB, asymptote {0 if w <= 2 else -40 * np.log10(w / 2):.0f} dB, phase {deg(g):.1f}")

speed plant at 0.025: exact -6.06 dB, phase -5.7 deg
speed plant at 0.25: exact -9.03 dB, phase -45.0 deg
speed plant at 2.5: exact -26.06 dB, phase -84.3 deg
speed plant at 25: exact -46.02 dB, phase -89.4 deg
heading loop at 0.2: exact 0.08 dB, asymptote 0 dB, phase -2.3
heading loop at 2: exact 7.96 dB, asymptote 0 dB, phase -90.0
heading loop at 20: exact -39.92 dB, asymptote -40 dB, phase -177.7
heading loop at 200: exact -80.00 dB, asymptote -80 dB, phase -179.8


## 7. Resonance (Note, Section 6)

In [8]:
g = G(2j)
print("at w_n = 2 rad/s:", g, "gain", abs(g), "phase", deg(g), " 1/(2 zeta) =", 1 / (2 * zeta))
x = wn**2 * (1 - 2 * zeta**2)
D = (wn**2 - x)**2 + (2 * zeta * wn)**2 * x
print("x =", round(x, 4), " w_r =", round(np.sqrt(x), 4), " D =", round(D, 4), " peak 4/sqrt(D) =", round(wn**2 / np.sqrt(D), 4),
      " formula", round(1 / (2 * zeta * np.sqrt(1 - zeta**2)), 4), " in dB", round(db(1 / (2 * zeta * np.sqrt(1 - zeta**2))), 2))
w = np.logspace(-2, 2, 400001)
m = np.abs(G(1j * w)); print("numerical peak", round(m.max(), 4), "at", round(w[m.argmax()], 4))
for z in [1, 0.707, 0.6, 0.5, 0.3, 0.2, 0.1]:
    m = np.abs(G(1j * w, KP, 0.4 * z))
    print(f"zeta {z}: Kd {0.4 * z:.3f}, peak {m.max():.3f} at {w[m.argmax()]:.3f} rad/s, 1/(2 zeta) = {1 / (2 * z):.2f}")

at w_n = 2 rad/s: -2.5j gain 2.5 phase -90.0  1/(2 zeta) = 2.5
x = 3.68  w_r = 1.9183  D = 2.4576  peak 4/sqrt(D) = 2.5516  formula 2.5516  in dB 8.14
numerical peak 2.5516 at 1.9183
zeta 1: Kd 0.400, peak 1.000 at 0.010 rad/s, 1/(2 zeta) = 0.50
zeta 0.707: Kd 0.283, peak 1.000 at 0.035 rad/s, 1/(2 zeta) = 0.71


zeta 0.6: Kd 0.240, peak 1.042 at 1.058 rad/s, 1/(2 zeta) = 0.83


zeta 0.5: Kd 0.200, peak 1.155 at 1.414 rad/s, 1/(2 zeta) = 1.00


zeta 0.3: Kd 0.120, peak 1.747 at 1.811 rad/s, 1/(2 zeta) = 1.67
zeta 0.2: Kd 0.080, peak 2.552 at 1.918 rad/s, 1/(2 zeta) = 2.50


zeta 0.1: Kd 0.040, peak 5.025 at 1.980 rad/s, 1/(2 zeta) = 5.00


## 8. Bandwidth (Note, Section 7)

In [9]:
def bandwidth(f):
    g0 = abs(f(1e-9j))
    return brentq(lambda x: abs(f(1j * x)) - g0 / np.sqrt(2), 1e-3, 1e3)
b = bandwidth(G); bt = bandwidth(lambda s: G(s, 0.2, 0.283)); bs = bandwidth(P_speed)
print("first try (Kp 0.4, Kd 0.08):", round(b, 3), "rad/s, period", round(2 * np.pi / b, 2), "s, phase there", round(deg(G(1j * b)), 1), "deg")
w = np.logspace(-2, 2, 400001)
print("RO-060's tuned gains (Kp 0.2, Kd 0.283): natural frequency", round(np.sqrt(0.2 / I), 3), "rad/s, damping ratio",
      round(0.283 / (2 * np.sqrt(0.2 * I)), 4), ", largest gain", round(np.abs(G(1j * w, 0.2, 0.283)).max(), 4), ", bandwidth", round(bt, 3), "rad/s")
print("speed plant:", round(bs, 4), "rad/s")
print("-3 dB is", round(db(1 / np.sqrt(2)), 2), "dB")

first try (Kp 0.4, Kd 0.08): 3.019 rad/s, period 2.08 s, phase there -154.7 deg


RO-060's tuned gains (Kp 0.2, Kd 0.283): natural frequency 1.414 rad/s, damping ratio 1.0006 , largest gain 0.9999 , bandwidth 0.909 rad/s
speed plant: 0.25 rad/s
-3 dB is -3.01 dB
